In [1]:
import pandas as pd
from pathlib import Path

from modules.direct_trust import (
    pearson_similarity,
    cosine_similarity,
    combined_similarity,
    get_top_k_neighbours,
    compute_direct_trust
)

DATA_DIR = Path("data")

rt_clean = pd.read_csv(DATA_DIR / "rt_clean.csv")

print("Dataset shape:", rt_clean.shape)
rt_clean.head()

Dataset shape: (339, 5826)


,0,1,2,3,4,5,6,7,8,9,...,5816,5817,5818,5819,5820,5821,5822,5823,5824,5825
0,0.700885,0.988599,0.988149,0.988949,0.988899,0.973649,0.977349,0.971699,0.980699,0.967548,...,0.602580,0.716336,0.519976,0.891395,0.729886,0.675684,0.737737,0.664533,0.661133,NaN
1,0.893495,0.986899,0.986349,0.987449,0.987299,0.978649,0.967398,0.967898,0.978049,0.963998,...,0.975049,0.991800,0.977149,0.985999,0.986499,0.985649,0.884594,0.987299,0.986849,NaN
2,0.957298,0.981699,0.981199,0.982149,0.982099,0.994250,0.967548,0.950998,0.967898,0.939347,...,0.959298,0.974199,0.960698,0.969498,NaN,NaN,0.477224,NaN,NaN,NaN
3,0.965348,0.988699,0.988349,0.988999,0.989049,0.982799,0.961748,0.960348,0.981449,0.972399,...,0.981199,0.987249,0.968948,0.991150,0.991650,0.992900,0.989149,0.991500,0.991350,NaN
4,0.956698,0.988649,0.988349,0.988999,0.988949,0.981699,0.958198,0.956848,0.981399,0.969048,...,0.979349,0.992500,0.991350,0.995450,0.971599,0.995400,0.991800,0.995500,0.995250,NaN


In [2]:
import pandas as pd
from pathlib import Path

DATA_DIR = Path("data")

rt_clean = pd.read_csv(DATA_DIR / "rt_clean.csv")

print("Before fixing:", rt_clean.shape)
print("First 5 column names:")
print(rt_clean.columns[:5])

Before fixing: (339, 5826)
First 5 column names:
Index(['0', '1', '2', '3', '4'], dtype='object')


In [3]:
print("Current shape:", rt_clean.shape)

print("Last 5 column names:")
print(rt_clean.columns[-5:])

print("Non-empty values in last 5 columns:")
print(rt_clean.iloc[:, -5:].notna().sum())

Current shape: (339, 5826)
Last 5 column names:
Index(['5821', '5822', '5823', '5824', '5825'], dtype='object')
Non-empty values in last 5 columns:
5821    330
5822    332
5823    332
5824    332
5825      0
dtype: int64


In [4]:
# Drop columns that are completely empty
rt_clean = rt_clean.dropna(axis=1, how="all")

print("After removing empty columns:", rt_clean.shape)

# Save corrected file
rt_clean.to_csv(DATA_DIR / "rt_clean.csv", index=False)

After removing empty columns: (339, 5818)


In [5]:
import pandas as pd
import numpy as np
from pathlib import Path

DATA_DIR = Path("data")
rt_path = DATA_DIR / "rtMatrix.txt"

# Load original raw file
rt_raw = pd.read_csv(rt_path, sep="\t", header=None)

print("Raw shape before fixing:", rt_raw.shape)

# If there is one extra column because of trailing tab, keep only first 5825 columns
if rt_raw.shape[1] > 5825:
    rt_raw = rt_raw.iloc[:, :5825]

print("Raw shape after fixing:", rt_raw.shape)

# Replace -1 with NaN
rt = rt_raw.replace(-1, np.nan)

# Normalise response time and convert to quality score
max_rt = rt.max().max()
rt_normalised = rt / max_rt

# Lower response time = better quality
rt_clean = 1 - rt_normalised

# Keep missing values as NaN
rt_clean = rt_clean.where(rt.notna(), np.nan)

# Save corrected clean file
rt_clean.to_csv(DATA_DIR / "rt_clean.csv", index=False)

print("Corrected rt_clean.csv saved.")
print("Final shape:", rt_clean.shape)
print("Missing values:", rt_clean.isna().sum().sum())
print("Minimum value:", rt_clean.min().min())
print("Maximum value:", rt_clean.max().max())

Raw shape before fixing: (339, 5826)
Raw shape after fixing: (339, 5825)
Corrected rt_clean.csv saved.
Final shape: (339, 5825)
Missing values: 100837
Minimum value: 0.0
Maximum value: 0.999949997499875


In [6]:
np.random.seed(42)

rt_malicious = rt_clean.copy()

num_users = rt_malicious.shape[0]
num_malicious_users = int(np.ceil(num_users * 0.10))

malicious_user_indices = np.random.choice(
    rt_malicious.index,
    size=num_malicious_users,
    replace=False
)

for user_id in malicious_user_indices:
    rated_items = rt_malicious.loc[user_id].notna()
    attack_type = np.random.choice(["high", "low"])
    
    if attack_type == "high":
        rt_malicious.loc[user_id, rated_items] = 1.0
    else:
        rt_malicious.loc[user_id, rated_items] = 0.0

rt_malicious.to_csv(DATA_DIR / "rt_malicious.csv", index=False)

labels = pd.DataFrame({
    "user_id": malicious_user_indices,
    "is_malicious": 1
})

labels.to_csv(DATA_DIR / "malicious_users_10_percent.csv", index=False)

print("Corrected rt_malicious.csv saved.")
print("Malicious shape:", rt_malicious.shape)
print("Malicious users:", num_malicious_users)

Corrected rt_malicious.csv saved.
Malicious shape: (339, 5825)
Malicious users: 34


In [7]:
rt_clean = pd.read_csv(DATA_DIR / "rt_clean.csv")

print("Dataset shape:", rt_clean.shape)

Dataset shape: (339, 5825)


In [8]:
import pandas as pd
from pathlib import Path

from modules.direct_trust import (
    pearson_similarity,
    cosine_similarity,
    combined_similarity,
    get_top_k_neighbours,
    compute_direct_trust
)

DATA_DIR = Path("data")

rt_clean = pd.read_csv(DATA_DIR / "rt_clean.csv")

print("Dataset shape:", rt_clean.shape)
rt_clean.head()


Dataset shape: (339, 5825)


,0,1,2,3,4,5,6,7,8,9,...,5815,5816,5817,5818,5819,5820,5821,5822,5823,5824
0,0.700885,0.988599,0.988149,0.988949,0.988899,0.973649,0.977349,0.971699,0.980699,0.967548,...,NaN,0.602580,0.716336,0.519976,0.891395,0.729886,0.675684,0.737737,0.664533,0.661133
1,0.893495,0.986899,0.986349,0.987449,0.987299,0.978649,0.967398,0.967898,0.978049,0.963998,...,NaN,0.975049,0.991800,0.977149,0.985999,0.986499,0.985649,0.884594,0.987299,0.986849
2,0.957298,0.981699,0.981199,0.982149,0.982099,0.994250,0.967548,0.950998,0.967898,0.939347,...,NaN,0.959298,0.974199,0.960698,0.969498,NaN,NaN,0.477224,NaN,NaN
3,0.965348,0.988699,0.988349,0.988999,0.989049,0.982799,0.961748,0.960348,0.981449,0.972399,...,NaN,0.981199,0.987249,0.968948,0.991150,0.991650,0.992900,0.989149,0.991500,0.991350
4,0.956698,0.988649,0.988349,0.988999,0.988949,0.981699,0.958198,0.956848,0.981399,0.969048,...,NaN,0.979349,0.992500,0.991350,0.995450,0.971599,0.995400,0.991800,0.995500,0.995250


In [9]:
target_user = 0
service_id = 0

neighbours = get_top_k_neighbours(
    rating_matrix=rt_clean,
    target_user=target_user,
    service_id=service_id,
    k=10,
    min_corated=3
)

print("Number of neighbours found:", len(neighbours))
print("Top neighbours:")
print(neighbours[:5])

Number of neighbours found: 10
Top neighbours:
[(10, 0.9040558720676246), (310, 0.9037891035382051), (221, 0.9017350720346355), (143, 0.8997571885358563), (283, 0.8993593987336288)]


In [10]:
dt_score = compute_direct_trust(
    rating_matrix=rt_clean,
    target_user=0,
    service_id=0,
    k=10,
    min_corated=3
)

print("Direct Trust Score:", dt_score)

Direct Trust Score: 0.646834264203931


In [11]:
RESULTS_DIR = Path("results")
RESULTS_DIR.mkdir(exist_ok=True)

direct_trust_results = []

for service_id in range(20):
    dt = compute_direct_trust(
        rating_matrix=rt_clean,
        target_user=0,
        service_id=service_id,
        k=10,
        min_corated=3
    )

    direct_trust_results.append({
        "user_id": 0,
        "service_id": service_id,
        "direct_trust": dt
    })

direct_trust_df = pd.DataFrame(direct_trust_results)

direct_trust_df.to_csv(
    RESULTS_DIR / "direct_trust_user_0_sample.csv",
    index=False
)

direct_trust_df

,user_id,service_id,direct_trust
0,0,0,0.646834
1,0,1,0.984484
2,0,2,0.983538
3,0,3,0.984934
4,0,4,0.984567
5,0,5,0.717565
6,0,6,0.678956
7,0,7,0.952599
8,0,8,0.972653
9,0,9,0.664982
